# Phase 1: Data Cleaning

Load the survey CSV, decode Borg emoji to 0-10, convert NASA-TLX text to numbers, save cleaned.csv.

## Step 0 — Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

CWD = Path.cwd()
ROOT = CWD if (CWD / 'data').exists() else CWD.parent
RAW = ROOT / 'data' / 'raw' / 'delivery_rider_survey.csv'
OUT = ROOT / 'data' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)

## Step 1 — Load and inspect

In [2]:
df_raw = pd.read_csv(RAW)
print(df_raw.shape)
df_raw.head()

(182, 48)


,Timestamp,Gender,Age,Education,Region,Marital status,Delivery platform,Job duration (experience),Monthly income,Type of vehicle,Mode of carrying deliveries,Working Hours per Day,Working days per week,Number of deliveries per day,Duration of rest break,Type of break,Tobacco Consumption,Alcohol Consumption,"In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Row 1Neck]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Shoulders]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Upper back]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Lower back]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Elbows]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Wrists / Hands]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Hips / Thighs]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Knees]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Ankles / Feet]","In the past 7 days, have you experienced discomfort in [Neck]","In the past 7 days, have you experienced discomfort in [Lower back]","In the past 7 days, have you experienced discomfort in [Knees]","In the past 7 days, have you experienced discomfort in [Wrist/Hands]",Has this discomfort reduced your delivery performance?,Have you ever taken leave due to body pain caused by delivery work?,Have you consulted a doctor or physiotherapist for work-related pain?,Does riding/driving for long hours increase your discomfort?,Does carrying heavy packages worsen your discomfort?,"How mentally demanding was your delivery shift (e.g., navigating routes, managing multiple orders, handling customer calls, using the app)? [.]","How physically demanding was your delivery shift (e.g., riding/driving long distances, climbing stairs, carrying packages, weather conditions)? [.]","How rushed or time-pressured did you feel while completing your deliveries (e.g., delivery deadlines, traffic delays, app timers)? [.]",How satisfied were you with your ability to complete deliveries accurately and on time during your shift? [.],"How much effort did you have to put in to maintain your delivery performance (e.g., speed, accuracy, customer interaction)? [.]","How stressed, frustrated, or emotionally strained did you feel during your delivery shift (e.g., due to customer behaviour, traffic, ratings, earnings pressure)? [.]","Overall, how hard did your delivery work feel today? [Your rating]",How tired did your legs feel while walking/riding? [Your rating],"How hard was it to breathe during deliveries (due to speed, heat, stairs, etc.)? [Your rating]",How hard did lifting or carrying parcels feel? [Your rating],How physically stressful was working in traffic/weather conditions? [Your rating],"At the end of your shift, how exhausted did your body feel? [Your rating]"
0,3/3/2026 0:41:28,Male,25-35,Degree/master’s degree or higher,Local,Married,Blinkit,<6 months,"10,000 - 19,999",Scooter,Bike storage / carrier,3-5 hrs,7,11-20,5-15 min,Uneven,Never,Never,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,75,25,25,75,75,00 ( Low ),🙁,🙂,😁 (Extremely Easy),😁 (Extremely Easy),🙁,🙂
1,3/3/2026 0:44:51,Female,25-35,High school,Local,Married,Blinkit,6 -12 months,"10,000 - 19,999",Scooter,Handheld,6-8 hrs,7,21-30,>30 min,Even,Never,Never,No,Yes,Yes,Yes,No,Yes,No,No,No,No,Yes,No,No,No,Yes,No,Yes,No,50,50,00 ( Low ),75,25,00 ( Low ),😣,🙂,🙁,😐,😭 (

In [3]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 182 entries, 0 to 181
Data columns (total 48 columns):
 #   Column                                                                                                                                                                 Non-Null Count  Dtype 
---  ------                                                                                                                                                                 --------------  ----- 
 0   Timestamp                                                                                                                                                              182 non-null    object
 1   Gender                                                                                                                                                                 182 non-null    object
 2   Age                                                                                                           

## Step 2 — Missing values and duplicates

In [4]:
print('missing:', df_raw.isna().sum().sum())
print('duplicates:', df_raw.duplicated().sum())

missing: 0
duplicates: 0


## Step 3 — Inspect categorical columns

In [5]:
for c in df_raw.columns[1:19]:
    print(c)
    print(' ', df_raw[c].value_counts(dropna=False).to_dict())

Gender
  {'Male': 152, 'Female': 30}
Age
  {'<25': 78, '25-35': 66, '36-45': 30, '>=46': 8}
Education
  {'Degree/master’s degree or higher': 125, 'High school': 49, 'Middle school or primary school': 8}
Region
  {'Local': 153, 'Non-Local ( not Tamil Nadu )': 29}
Marital status
  {'Unmarried': 103, 'Married': 79}
Delivery platform
  {'Blinkit': 97, 'Zepto': 80, 'Both': 5}
Job duration (experience)
  {'6 -12 months': 70, '<6 months': 64, '12 - 24 months': 33, '>24 months': 15}
Monthly income
  {'20,000 - 35,000': 66, '10,000 - 19,999': 59, '>35,000': 37, '<10,000': 20}
Type of vehicle
  {'Scooter': 94, 'Motor bike': 88}
Mode of carrying deliveries
  {'Backpack': 108, 'Bike storage / carrier': 50, 'Handheld': 16, 'Bike handle': 8}
Working Hours per Day
  {'> 8 hrs': 89, '6-8 hrs': 56, '3-5 hrs': 35, '< 3 hrs': 2}
Working days per week
  {'7': 80, '5-6': 66, '3-4': 32, '< 3': 4}
Number of deliveries per day
  {'>=31': 68, '21-30': 57, '11-20': 55, '<=10': 2}
Duration of rest break
  {'<5 m

## Step 4 — Inspect NASA-TLX columns

In [6]:
# 6 NASA-TLX items: columns ending with '[.]'
nasa_cols = [c for c in df_raw.columns if c.endswith('[.]')]
for c in nasa_cols:
    print(c[:60])
    print(' ', df_raw[c].value_counts(dropna=False).to_dict())

How mentally demanding was your delivery shift (e.g., naviga
  {'50': 74, '25': 53, '75': 28, '100 ( High )': 15, '00 ( low )': 12}
How physically demanding was your delivery shift (e.g., ridi
  {'50': 68, '75': 38, '100 ( High )': 34, '25': 30, '00 ( Low )': 12}
How rushed or time-pressured did you feel while completing y
  {'25': 60, '50': 50, '00 ( Low )': 34, '75': 20, '100 ( High )': 18}
How satisfied were you with your ability to complete deliver
  {'75': 69, '50': 49, '100 ( High )': 43, '25': 16, '00 ( Low )': 5}
How much effort did you have to put in to maintain your deli
  {'100 ( High )': 75, '50': 54, '75': 31, '25': 13, '00 ( Low )': 9}
How stressed, frustrated, or emotionally strained did you fe
  {'50': 57, '25': 37, '75': 36, '100 ( High )': 36, '00 ( Low )': 16}


## Step 5 — Inspect Borg CR10 emoji columns

In [7]:
# Last 6 columns are the Borg items. Print codepoints to build the decode map.
borg_cols = list(df_raw.columns[-6:])
for c in borg_cols:
    print(c[:60])
    for v, n in df_raw[c].value_counts(dropna=False).items():
        cps = ' '.join(f'U+{ord(ch):04X}' for ch in str(v))
        print(f'  n={n}  {v!r}  [{cps}]')

Overall, how hard did your delivery work feel today? [Your r
  n=45  '🙂'  [U+1F642]
  n=26  '😐'  [U+1F610]
  n=24  '😊'  [U+1F60A]
  n=23  '🙁'  [U+1F641]
  n=19  '☹️'  [U+2639 U+FE0F]
  n=12  '😣'  [U+1F623]
  n=11  '😁 (Extremely Easy)'  [U+1F601 U+0020 U+0028 U+0045 U+0078 U+0074 U+0072 U+0065 U+006D U+0065 U+006C U+0079 U+0020 U+0045 U+0061 U+0073 U+0079 U+0029]
  n=7  '😖'  [U+1F616]
  n=7  '😭 (Extremely Hard)'  [U+1F62D U+0020 U+0028 U+0045 U+0078 U+0074 U+0072 U+0065 U+006D U+0065 U+006C U+0079 U+0020 U+0048 U+0061 U+0072 U+0064 U+0029]
  n=4  '😩'  [U+1F629]
  n=4  '😫'  [U+1F62B]
How tired did your legs feel while walking/riding? [Your rat
  n=34  '😐'  [U+1F610]
  n=32  '🙂'  [U+1F642]
  n=22  '☹️'  [U+2639 U+FE0F]
  n=22  '🙁'  [U+1F641]
  n=16  '😊'  [U+1F60A]
  n=16  '😖'  [U+1F616]
  n=11  '😫'  [U+1F62B]
  n=10  '😁 (Extremely Easy)'  [U+1F601 U+0020 U+0028 U+0045 U+0078 U+0074 U+0072 U+0065 U+006D U+0065 U+006C U+0079 U+0020 U+0045 U+0061 U+0073 U+0079 U+0029]
  n=10  '😣'  [U+1F623]


## Step 6 — Fix mojibake / curly apostrophe

In [8]:
df = df_raw.copy()

# Curly apostrophe in Education -> plain
df['Education'] = df['Education'].str.replace(chr(0x2019), chr(0x27), regex=False)
df['Education'].unique()

array(["Degree/master's degree or higher", 'High school',
       'Middle school or primary school'], dtype=object)

## Step 7 — Trim whitespace

In [9]:
df.columns = df.columns.str.strip()
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].str.strip()

## Step 8 — Decode Borg to 0-10

In [10]:
# 0 = Extremely Easy ... 10 = Extremely Hard (face order from the Google Form)
borg_map = {
    chr(0x1F601): 0,
    chr(0x1F60A): 1,
    chr(0x1F642): 2,
    chr(0x1F610): 3,
    chr(0x1F641): 4,
    chr(0x2639):  5,
    chr(0x1F623): 6,
    chr(0x1F616): 7,
    chr(0x1F629): 8,
    chr(0x1F62B): 9,
    chr(0x1F62D): 10,
}

# Endpoint cells carry trailing text like '(Extremely Easy)', so key by first char
borg_cols = list(df.columns[-6:])
for c in borg_cols:
    df[c] = df[c].apply(lambda v: borg_map[str(v)[0]]).astype('int8')

df[borg_cols].describe()

,"Overall, how hard did your delivery work feel today? [Your rating]",How tired did your legs feel while walking/riding? [Your rating],"How hard was it to breathe during deliveries (due to speed, heat, stairs, etc.)? [Your rating]",How hard did lifting or carrying parcels feel? [Your rating],How physically stressful was working in traffic/weather conditions? [Your rating],"At the end of your shift, how exhausted did your body feel? [Your rating]"
count,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000
mean,3.505495,4.005495,4.357143,4.148352,6.126374,5.038462
std,2.464661,2.521723,2.798762,2.645746,3.050309,3.346253
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,2.000000,2.000000,2.000000,4.000000,2.000000
50%,3.000000,3.000000,4.000000,4.000000,6.000000,5.000000
75%,5.000000,5.750000,6.000000,6.000000,9.000000,8.000000
max,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000


## Step 9 — Convert NASA-TLX to numeric 0-100

In [11]:
# '00 ( Low )' / '100 ( High )' / '50' -> leading number
nasa_cols = [c for c in df.columns if c.endswith('[.]')]
for c in nasa_cols:
    df[c] = df[c].astype(str).str.strip().str.split().str[0].astype(int)

df[nasa_cols].describe()

,"How mentally demanding was your delivery shift (e.g., navigating routes, managing multiple orders, handling customer calls, using the app)? [.]","How physically demanding was your delivery shift (e.g., riding/driving long distances, climbing stairs, carrying packages, weather conditions)? [.]","How rushed or time-pressured did you feel while completing your deliveries (e.g., delivery deadlines, traffic delays, app timers)? [.]",How satisfied were you with your ability to complete deliveries accurately and on time during your shift? [.],"How much effort did you have to put in to maintain your delivery performance (e.g., speed, accuracy, customer interaction)? [.]","How stressed, frustrated, or emotionally strained did you feel during your delivery shift (e.g., due to customer behaviour, traffic, ratings, earnings pressure)? [.]"
count,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000
mean,47.390110,57.142857,40.109890,67.719780,70.604396,55.357143
std,25.412106,28.615790,29.935271,25.313363,29.753454,30.682997
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,25.000000,50.000000,25.000000,50.000000,50.000000,25.000000
50%,50.000000,50.000000,25.000000,75.000000,75.000000,50.000000
75%,50.000000,75.000000,50.000000,75.000000,100.000000,75.000000
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


## Step 10 — Validate and save

In [12]:
assert df.shape == df_raw.shape
assert df.isna().sum().sum() == 0

out = OUT / 'cleaned.csv'
df.to_csv(out, index=False)
print('saved', out, df.shape)
df.head()

saved f:\Ergonomic_Project\data\processed\cleaned.csv (182, 48)


,Timestamp,Gender,Age,Education,Region,Marital status,Delivery platform,Job duration (experience),Monthly income,Type of vehicle,Mode of carrying deliveries,Working Hours per Day,Working days per week,Number of deliveries per day,Duration of rest break,Type of break,Tobacco Consumption,Alcohol Consumption,"In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Row 1Neck]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Shoulders]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Upper back]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Lower back]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Elbows]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Wrists / Hands]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Hips / Thighs]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Knees]","In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Ankles / Feet]","In the past 7 days, have you experienced discomfort in [Neck]","In the past 7 days, have you experienced discomfort in [Lower back]","In the past 7 days, have you experienced discomfort in [Knees]","In the past 7 days, have you experienced discomfort in [Wrist/Hands]",Has this discomfort reduced your delivery performance?,Have you ever taken leave due to body pain caused by delivery work?,Have you consulted a doctor or physiotherapist for work-related pain?,Does riding/driving for long hours increase your discomfort?,Does carrying heavy packages worsen your discomfort?,"How mentally demanding was your delivery shift (e.g., navigating routes, managing multiple orders, handling customer calls, using the app)? [.]","How physically demanding was your delivery shift (e.g., riding/driving long distances, climbing stairs, carrying packages, weather conditions)? [.]","How rushed or time-pressured did you feel while completing your deliveries (e.g., delivery deadlines, traffic delays, app timers)? [.]",How satisfied were you with your ability to complete deliveries accurately and on time during your shift? [.],"How much effort did you have to put in to maintain your delivery performance (e.g., speed, accuracy, customer interaction)? [.]","How stressed, frustrated, or emotionally strained did you feel during your delivery shift (e.g., due to customer behaviour, traffic, ratings, earnings pressure)? [.]","Overall, how hard did your delivery work feel today? [Your rating]",How tired did your legs feel while walking/riding? [Your rating],"How hard was it to breathe during deliveries (due to speed, heat, stairs, etc.)? [Your rating]",How hard did lifting or carrying parcels feel? [Your rating],How physically stressful was working in traffic/weather conditions? [Your rating],"At the end of your shift, how exhausted did your body feel? [Your rating]"
0,3/3/2026 0:41:28,Male,25-35,Degree/master's degree or higher,Local,Married,Blinkit,<6 months,"10,000 - 19,999",Scooter,Bike storage / carrier,3-5 hrs,7,11-20,5-15 min,Uneven,Never,Never,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,75,25,25,75,75,0,4,2,0,0,4,2
1,3/3/2026 0:44:51,Female,25-35,High school,Local,Married,Blinkit,6 -12 months,"10,000 - 19,999",Scooter,Handheld,6-8 hrs,7,21-30,>30 min,Even,Never,Never,No,Yes,Yes,Yes,No,Yes,No,No,No,No,Yes,No,No,No,Yes,No,Yes,No,50,50,0,75,25,0,6,2,4,3,10,7
2,3/3/2026 0:47:05,Male,<25,Degree/master's degree or highe